In [1]:
#Text preprocessing

In [5]:
# Imports + Load preprocessed dataset + SpaCy setup

import re
import os
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed/reviews_processed.csv")
df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# ---- SpaCy setup ----
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
    print("SpaCy model loaded: en_core_web_sm")
except Exception as e:
    print("SpaCy model NOT loaded:", e)
    print("If needed, install once in a terminal:")
    print("python -m spacy download en_core_web_sm")

Loaded: D:\University of Bicocca\Data science\2nd year 2025-26\1st Semester\Text Mining & Search\Project\drug-review-sentiment-topic-modeling\data\processed\reviews_processed.csv
Shape: (50000, 4)
Columns: ['review', 'rating', 'condition', 'sentiment']
SpaCy model loaded: en_core_web_sm


In [7]:
#  Baseline snapshot 

print("===== BASELINE  =====")
print("Shape:", df.shape)
print("Missing review:", df["review"].isna().mean())

# Fixed random sample (reproducible)
sample_idx = df.sample(3, random_state=42).index
print("\n3 sample raw reviews (fixed seed):")
for i in sample_idx:
    print("\n--- index:", i)
    print(df.loc[i, "review"][:500])  # truncate for readability

# Noise scan metrics
s = df["review"].astype(str)

pct_has_angle = s.str.contains(r"[<>]", regex=True).mean()
pct_has_entities = s.str.contains(r"&amp;|&#", regex=True).mean()
pct_has_digits = s.str.contains(r"\d", regex=True).mean()
pct_has_nonascii = s.apply(lambda x: any(ord(ch) > 127 for ch in x)).mean()

print("\nNoise scan (% of reviews):")
print(f"Contains < or > (HTML tags): {pct_has_angle:.4f}")
print(f"Contains &amp; or &# (entities): {pct_has_entities:.4f}")
print(f"Contains digits: {pct_has_digits:.4f}")
print(f"Contains non-ascii chars: {pct_has_nonascii:.4f}")

print("=====================================")

===== BASELINE  =====
Shape: (50000, 4)
Missing review: 0.0

3 sample raw reviews (fixed seed):

--- index: 33553
"The Best."

--- index: 9427
"I've had this medication for almost 5 months now. I switched from the mirena IUD. The doctor told me it would be perfect so I wouldn't have to worry about getting pregnant and taking it. I liked it the first month. After the bruising went down I never thought of it, till the second month came. When I was on the IUD, I never had a period besides the first month. Now, my periods are a dark brown (sorry still grosses me out) and really heavy. They last 3-4 days and come back in two weeks or so. I'

--- index: 199
"This drug is not for me. Not only did I have bad dreams and hallucinations, it seriously affected my memory. The four months I took it I didn't think anything was wrong. My family had to tell me how goofy (but funny and nice) I was acting...very weird, I'm not that nice.  Still waiting for those 4 months of memory to return. Also, my doc

In [8]:
#  Lowercasing

df["review_lower"] = df["review"].str.lower()

# Show 5 examples (same fixed indices for consistency)
print("===== LOWERCASE TRANSFORMATION =====")

for i in sample_idx[:3]:
    print("\n--- index:", i)
    print("BEFORE:\n", df.loc[i, "review"][:300])
    print("AFTER:\n", df.loc[i, "review_lower"][:300])

# Metric: % reviews containing uppercase letters BEFORE vs AFTER
pct_upper_before = df["review"].str.contains(r"[A-Z]").mean()
pct_upper_after = df["review_lower"].str.contains(r"[A-Z]").mean()

print("\nUppercase presence:")
print(f"Before: {pct_upper_before:.4f}")
print(f"After:  {pct_upper_after:.4f}")

print("=====================================")

===== LOWERCASE TRANSFORMATION =====

--- index: 33553
BEFORE:
 "The Best."
AFTER:
 "the best."

--- index: 9427
BEFORE:
 "I've had this medication for almost 5 months now. I switched from the mirena IUD. The doctor told me it would be perfect so I wouldn't have to worry about getting pregnant and taking it. I liked it the first month. After the bruising went down I never thought of it, till the second month came. When
AFTER:
 "i've had this medication for almost 5 months now. i switched from the mirena iud. the doctor told me it would be perfect so i wouldn't have to worry about getting pregnant and taking it. i liked it the first month. after the bruising went down i never thought of it, till the second month came. when

--- index: 199
BEFORE:
 "This drug is not for me. Not only did I have bad dreams and hallucinations, it seriously affected my memory. The four months I took it I didn't think anything was wrong. My family had to tell me how goofy (but funny and nice) I was acting...v

In [9]:
# Remove HTML tags

def remove_html_tags(text):
    return re.sub(r"<.*?>", " ", text)

df["review_no_html"] = df["review_lower"].apply(remove_html_tags)

print("===== HTML TAG REMOVAL =====")

# % containing < or > before vs after
pct_tags_before = df["review_lower"].str.contains(r"[<>]").mean()
pct_tags_after = df["review_no_html"].str.contains(r"[<>]").mean()

print(f"Contains < or > BEFORE: {pct_tags_before:.4f}")
print(f"Contains < or > AFTER:  {pct_tags_after:.4f}")

# Show examples where tags existed (if any)
tag_examples = df[df["review_lower"].str.contains(r"[<>]")].head(3)

for i in tag_examples.index:
    print("\n--- index:", i)
    print("BEFORE:\n", df.loc[i, "review_lower"][:300])
    print("AFTER:\n", df.loc[i, "review_no_html"][:300])

print("=====================================")

===== HTML TAG REMOVAL =====
Contains < or > BEFORE: 0.0014
Contains < or > AFTER:  0.0014

--- index: 836
BEFORE:
 "if i could rate 0 i would 
****warning please listen****
if you are considering the depo shot and have bipolar depression please be aware of the side effects that may occur!!! =====> extreme mood swings, irritably, feelings of familiarity, odd sensations and worse depression. if you are consideri
AFTER:
 "if i could rate 0 i would 
****warning please listen****
if you are considering the depo shot and have bipolar depression please be aware of the side effects that may occur!!! =====> extreme mood swings, irritably, feelings of familiarity, odd sensations and worse depression. if you are consideri

--- index: 1420
BEFORE:
 "ent prescribed me 1 x 25mg daily for 10 days to treat tinnitus, in case it was caused by a blocked eustachian tube.  i had a very bad reaction to steroid.  i suffered from severe insomnia, depression and anxiety.  psychologically, it caused me a lot o

In [10]:
# Remove punctuation / special characters (keep apostrophes)

def remove_punct(text):
    # keep letters, digits, apostrophes, and spaces
    return re.sub(r"[^a-z0-9'\s]", " ", text)

df["review_no_punct"] = df["review_no_html"].apply(remove_punct)

print("===== PUNCTUATION REMOVAL =====")

# Show 3 before/after examples (same fixed sample indices)
for i in sample_idx:
    print("\n--- index:", i)
    print("BEFORE:\n", df.loc[i, "review_no_html"][:300])
    print("AFTER:\n", df.loc[i, "review_no_punct"][:300])

# Token count change
tokens_before = df["review_no_html"].str.split().str.len().mean()
tokens_after = df["review_no_punct"].str.split().str.len().mean()

print("\nAverage token count:")
print("Before:", round(tokens_before, 2))
print("After: ", round(tokens_after, 2))

print("=================================")

===== PUNCTUATION REMOVAL =====

--- index: 33553
BEFORE:
 "the best."
AFTER:
  the best  

--- index: 9427
BEFORE:
 "i've had this medication for almost 5 months now. i switched from the mirena iud. the doctor told me it would be perfect so i wouldn't have to worry about getting pregnant and taking it. i liked it the first month. after the bruising went down i never thought of it, till the second month came. when
AFTER:
  i've had this medication for almost 5 months now  i switched from the mirena iud  the doctor told me it would be perfect so i wouldn't have to worry about getting pregnant and taking it  i liked it the first month  after the bruising went down i never thought of it  till the second month came  when

--- index: 199
BEFORE:
 "this drug is not for me. not only did i have bad dreams and hallucinations, it seriously affected my memory. the four months i took it i didn't think anything was wrong. my family had to tell me how goofy (but funny and nice) i was acting...very w

In [11]:
# Normalize whitespace

def normalize_whitespace(text):
    text = re.sub(r"\s+", " ", text)   # collapse any whitespace
    return text.strip()

df["review_norm_ws"] = df["review_no_punct"].apply(normalize_whitespace)

print("===== WHITESPACE NORMALIZATION =====")

# % reviews containing double spaces before vs after
pct_double_before = df["review_no_punct"].str.contains(r"\s{2,}").mean()
pct_double_after = df["review_norm_ws"].str.contains(r"\s{2,}").mean()

print(f"Contains double spaces BEFORE: {pct_double_before:.4f}")
print(f"Contains double spaces AFTER:  {pct_double_after:.4f}")

# Show same fixed examples
for i in sample_idx:
    print("\n--- index:", i)
    print("BEFORE:\n", df.loc[i, "review_no_punct"][:250])
    print("AFTER:\n", df.loc[i, "review_norm_ws"][:250])

print("====================================")

===== WHITESPACE NORMALIZATION =====
Contains double spaces BEFORE: 0.9872
Contains double spaces AFTER:  0.0000

--- index: 33553
BEFORE:
  the best  
AFTER:
 the best

--- index: 9427
BEFORE:
  i've had this medication for almost 5 months now  i switched from the mirena iud  the doctor told me it would be perfect so i wouldn't have to worry about getting pregnant and taking it  i liked it the first month  after the bruising went down i nev
AFTER:
 i've had this medication for almost 5 months now i switched from the mirena iud the doctor told me it would be perfect so i wouldn't have to worry about getting pregnant and taking it i liked it the first month after the bruising went down i never th

--- index: 199
BEFORE:
  this drug is not for me  not only did i have bad dreams and hallucinations  it seriously affected my memory  the four months i took it i didn't think anything was wrong  my family had to tell me how goofy  but funny and nice  i was acting   very wei
AFTER:
 this drug i

In [12]:
# Stopword removal (preserve negations)

from spacy.lang.en.stop_words import STOP_WORDS

NEGATIONS = {"no", "not", "nor", "never", "n't"}

# Stopwords minus negations
STOPWORDS_CUSTOM = set(STOP_WORDS) - NEGATIONS

def remove_stopwords_keep_neg(text):
    tokens = text.split()
    kept = [t for t in tokens if t not in STOPWORDS_CUSTOM]
    return " ".join(kept)

df["review_no_stop"] = df["review_norm_ws"].apply(remove_stopwords_keep_neg)

print("===== STOPWORD REMOVAL (NEGATIONS PRESERVED) =====")

# 5 examples: tokens before vs after (use 5 fixed indices)
example_idx = df.sample(5, random_state=42).index

for i in example_idx:
    before_toks = df.loc[i, "review_norm_ws"].split()[:25]
    after_toks = df.loc[i, "review_no_stop"].split()[:25]
    print("\n--- index:", i)
    print("BEFORE TOKENS (first 25):", before_toks)
    print("AFTER  TOKENS (first 25):", after_toks)

# Avg token count before vs after
avg_before = df["review_norm_ws"].str.split().str.len().mean()
avg_after  = df["review_no_stop"].str.split().str.len().mean()

print("\nAverage token count:")
print("Before:", round(avg_before, 2))
print("After: ", round(avg_after, 2))

# Negation preservation check
def count_negations(text):
    toks = text.split()
    return sum(t in {"no", "not", "never", "nor"} for t in toks) + sum("n't" in t for t in toks)

neg_before = df["review_norm_ws"].apply(count_negations).sum()
neg_after  = df["review_no_stop"].apply(count_negations).sum()

print("\nNegation occurrences (corpus-level):")
print("Before:", neg_before)
print("After: ", neg_after)

print("=================================================")

===== STOPWORD REMOVAL (NEGATIONS PRESERVED) =====

--- index: 33553
BEFORE TOKENS (first 25): ['the', 'best']
AFTER  TOKENS (first 25): ['best']

--- index: 9427
BEFORE TOKENS (first 25): ["i've", 'had', 'this', 'medication', 'for', 'almost', '5', 'months', 'now', 'i', 'switched', 'from', 'the', 'mirena', 'iud', 'the', 'doctor', 'told', 'me', 'it', 'would', 'be', 'perfect', 'so', 'i']
AFTER  TOKENS (first 25): ["i've", 'medication', '5', 'months', 'switched', 'mirena', 'iud', 'doctor', 'told', 'perfect', "wouldn't", 'worry', 'getting', 'pregnant', 'taking', 'liked', 'month', 'bruising', 'went', 'never', 'thought', 'till', 'second', 'month', 'came']

--- index: 199
BEFORE TOKENS (first 25): ['this', 'drug', 'is', 'not', 'for', 'me', 'not', 'only', 'did', 'i', 'have', 'bad', 'dreams', 'and', 'hallucinations', 'it', 'seriously', 'affected', 'my', 'memory', 'the', 'four', 'months', 'i', 'took']
AFTER  TOKENS (first 25): ['drug', 'not', 'not', 'bad', 'dreams', 'hallucinations', 'seriously'

In [13]:
#Lemmatization (SpaCy) + clean_review

NEG_KEEP = {"no", "not", "nor", "never"}  # explicit keep even if short
MIN_LEN = 2

def spacy_lemmatize(text):
    doc = nlp(text)
    lemmas = []
    for token in doc:
        if token.is_space:
            continue
        
        lemma = token.lemma_.lower().strip()

        # drop empty / pronoun artifacts
        if not lemma or lemma == "-pron-":
            continue

        # keep digits as-is
        if lemma.isdigit():
            lemmas.append(lemma)
            continue

        # keep negations even if short
        if lemma in NEG_KEEP:
            lemmas.append(lemma)
            continue

        # drop very short tokens
        if len(lemma) < MIN_LEN:
            continue

        # drop pure punctuation (should already be removed, but safe)
        if all(ch in "' " for ch in lemma):
            continue

        lemmas.append(lemma)

    return " ".join(lemmas)

# Apply lemmatization
df["clean_review"] = df["review_no_stop"].apply(spacy_lemmatize)

print("===== LEMMATIZATION (clean_review) =====")

# 5 examples before/after
example_idx2 = df.sample(5, random_state=42).index

for i in example_idx2:
    print("\n--- index:", i)
    print("BEFORE (no_stop):", df.loc[i, "review_no_stop"][:250])
    print("AFTER  (clean):  ", df.loc[i, "clean_review"][:250])

# Show a small sample of changed forms (heuristic: where token differs from lemma)

pairs = []
subset_texts = df["review_no_stop"].sample(200, random_state=42).tolist()

for text in subset_texts:
    doc = nlp(text)
    for tok in doc:
        if tok.is_space:
            continue
        w = tok.text.lower()
        l = tok.lemma_.lower()
        if w != l and w.isalpha() and l.isalpha():
            pairs.append((w, l))

# most common changes
from collections import Counter
common_changes = Counter(pairs).most_common(15)

print("\nTop changed forms (sample):")
for (w, l), c in common_changes:
    print(f"{w} -> {l}  (count={c})")

print("=======================================")

===== LEMMATIZATION (clean_review) =====

--- index: 33553
BEFORE (no_stop): best
AFTER  (clean):   well

--- index: 9427
BEFORE (no_stop): i've medication 5 months switched mirena iud doctor told perfect wouldn't worry getting pregnant taking liked month bruising went never thought till second month came iud never period month periods dark brown sorry grosses heavy 3 4 days come weeks i
AFTER  (clean):   have medication 5 month switch mirena iud doctor tell perfect would not worry get pregnant taking like month bruising go never think till second month come iud never period month period dark brown sorry gross heavy 3 4 day come week have experience t

--- index: 199
BEFORE (no_stop): drug not not bad dreams hallucinations seriously affected memory months took didn't think wrong family tell goofy funny nice acting weird i'm not nice waiting 4 months memory return doctor prescribed chantix stop smoking time probably combination lan
AFTER  (clean):   drug not not bad dream hallucination 

In [14]:
# Final validation

print("===== FINAL VALIDATION =====")

# Check existence
print("clean_review exists:", "clean_review" in df.columns)

# Null check
null_count = df["clean_review"].isna().sum()
print("Null clean_review:", null_count)

# Empty string check
empty_mask = df["clean_review"].str.strip() == ""
empty_count = empty_mask.sum()
print("Empty clean_review:", empty_count)

# Drop empty if any
if empty_count > 0:
    df = df[~empty_mask].copy()
    print("Dropped empty rows.")
else:
    print("No empty rows dropped.")

print("\nFinal row count:", len(df))

# Average length
avg_len_clean = df["clean_review"].str.split().str.len().mean()
print("Average clean_review token length:", round(avg_len_clean, 2))

# Preview 5 rows
preview_idx = df.sample(5, random_state=42).index
print("\nPreview review vs clean_review:")
for i in preview_idx:
    print("\n--- index:", i)
    print("RAW:   ", df.loc[i, "review"][:200])
    print("CLEAN: ", df.loc[i, "clean_review"][:200])

print("=================================")

===== FINAL VALIDATION =====
clean_review exists: True
Null clean_review: 0
Empty clean_review: 2
Dropped empty rows.

Final row count: 49998
Average clean_review token length: 39.59

Preview review vs clean_review:

--- index: 33553
RAW:    "The Best."
CLEAN:  well

--- index: 9427
RAW:    "I've had this medication for almost 5 months now. I switched from the mirena IUD. The doctor told me it would be perfect so I wouldn't have to worry about getting pregnant and taking it. I liked it t
CLEAN:  have medication 5 month switch mirena iud doctor tell perfect would not worry get pregnant taking like month bruising go never think till second month come iud never period month period dark brown sor

--- index: 199
RAW:    "This drug is not for me. Not only did I have bad dreams and hallucinations, it seriously affected my memory. The four months I took it I didn't think anything was wrong. My family had to tell me how 
CLEAN:  drug not not bad dream hallucination seriously affect memory mont

In [15]:
# save

OUT_PATH = Path("../data/processed/reviews_preprocessed.csv")

# Keep minimal columns for modeling
df_final = df[["review", "clean_review", "rating", "sentiment", "condition"]].copy()

df_final.to_csv(OUT_PATH, index=False)

print("Saved to:", OUT_PATH.resolve())
print("Final shape:", df_final.shape)
print("Columns:", list(df_final.columns))

Saved to: D:\University of Bicocca\Data science\2nd year 2025-26\1st Semester\Text Mining & Search\Project\drug-review-sentiment-topic-modeling\data\processed\reviews_preprocessed.csv
Final shape: (49998, 5)
Columns: ['review', 'clean_review', 'rating', 'sentiment', 'condition']
